# AutoGen Selector Teams

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the mechanics of AutoGen's `SelectorGroupChat`. We focus on managing conversational flow and preventing infinite loops.

We will cover 3 patterns:
1. **The Selector Router:** A central LLM dynamically choosing the next speaker based on context gaps.
2. **Circular Delegation Anti-Pattern:** Simulating an infinite loop and applying a hard termination constraint.
3. **The Single Agent Baseline:** Comparing latency/tokens of a single agent vs a team.

---
## Pattern 1: The Selector Router

The Selector evaluates the shared context and picks the next agent. We simulate a strict routing decision.

In [ ]:
def simulate_selector(context):
    print(f"\n🚦 [Selector LLM] Evaluating context: {context}")
    
    if "metrics" not in context:
        print("  -> Gap: Missing metrics. Selecting Telemetry Agent.")
        return "telemetry_agent"
    elif "hypothesis" not in context:
        print("  -> Gap: Missing hypothesis. Selecting Analyst Agent.")
        return "analyst_agent"
    else:
        print("  -> Gap: None. Selecting Reviewer Agent to finalize.")
        return "reviewer_agent"

# Simulating the flow
context = []
next_speaker = simulate_selector(context)

context.append("metrics: CPU 99%")
next_speaker = simulate_selector(context)

context.append("hypothesis: Rogue query")
next_speaker = simulate_selector(context)


---
## Pattern 2: Circular Delegation (The Infinite Loop)

Agents can argue forever. We simulate a loop and how `MaxMessageTermination` stops it.

In [ ]:
def simulate_argument_loop():
    print("\n🔄 Starting Agent Debate...")
    messages = 0
    max_messages = 4  # Simulating MaxMessageTermination(4)
    
    while True:
        messages += 1
        print(f"  [Msg {messages}] Analyst: The issue is the DB!")
        
        if messages >= max_messages:
            print("  🛑 [System] MaxMessageTermination reached. Halting loop.")
            break
            
        messages += 1
        print(f"  [Msg {messages}] Reviewer: I disagree, prove it!")
        
        if messages >= max_messages:
            print("  🛑 [System] MaxMessageTermination reached. Halting loop.")
            break

simulate_argument_loop()


---
## Pattern 3: The Single Agent Baseline

A team of 5 agents processes the context window 5 times. A single agent processes it once. We simulate the latency tax.

In [ ]:
def single_agent_run():
    start = time.time()
    print("\n[Single Agent] Reading context... diagnosing... outputting.")
    time.sleep(0.5)
    return time.time() - start

def selector_team_run():
    start = time.time()
    print("\n[Selector] Reading context... selecting DB Agent.")
    time.sleep(0.3)
    print("[DB Agent] Reading context... outputting metrics.")
    time.sleep(0.3)
    print("[Selector] Reading context... selecting Analyst.")
    time.sleep(0.3)
    print("[Analyst] Reading context... outputting hypothesis.")
    time.sleep(0.3)
    return time.time() - start

print(f"Single Agent Latency: {single_agent_run():.2f} seconds")
print(f"Selector Team Latency: {selector_team_run():.2f} seconds")
print("Conclusion: Only use the Team if you need adversarial Reviewers.")
